In [0]:
df = spark.read.csv("/Volumes/bigdata2/my_voloum/volume/movies.csv" , header= True , schema=movies_schema)
df2 = spark.read.csv("/Volumes/bigdata2/my_voloum/volume/ratings.csv" , header= True , schema=ratings_schema)

In [0]:
display(df)

In [0]:
display(df2)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, LongType
import pyspark.sql.functions as f

# Movies Schema
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

# Ratings Schema
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

In [0]:
df.printSchema()
df2.printSchema()

In [0]:
df = df.withColumn(
    "genre_array",
    f.split(f.col("genres"), "\\|")
)

In [0]:
df.select("title", "genre_array").show(5, False)

In [0]:
movies_exploded = df.withColumn(
    "genre",
    f.explode(f.col("genre_array"))
)

In [0]:
comedy_movies = movies_exploded.filter(
    f.col("genre") == "Comedy"
)

comedy_movies.select("movieId", "title", "genre").show()

In [0]:
joined_df = ratings_df.join(
    comedy_movies,
    on="movieId",
    how="inner"
)

joined_df.select("userId", "title", "rating").show()

In [0]:
joined_df.printSchema()

In [0]:
ratings_df = ratings_df.withColumn(
    "rating_time",
    f.from_unixtime("timestamp").cast("timestamp")
)

ratings_df.select("timestamp", "rating_time").show(5, False)